# GPU profiling and batch scaling

Select **T4 GPU** (or another available GPU) under **Runtime > Change runtime
type**, then run the cells in order. This study measures the current optimized
model at batch sizes **1, 2, 4 and 8**. It uses Colab's existing TensorFlow
installation and downloads no ForTy data.

Each batch repeats one synthetic tile and loads the same initial checkpoint.
Three rounds alternate batch order. Inference is float32; training is mixed
float16. Each measurement runs in a fresh process with five warmups and 30
steps. A batch above one that runs out of memory is marked incomplete and
excluded from throughput comparisons; other failures stop the study.

After timing, separate fresh processes capture three profiler steps at batch
one and the largest fully measured batch for each workload. Profiled steps
are excluded from latency and allocator-memory results. Raw traces are retained
for inspection; a saved trace alone does not prove GPU kernel coverage.

This measures synthetic compute capacity, not segmentation accuracy, training
convergence, file-to-file throughput or total GPU VRAM. Free GPU availability
and session duration vary. Download the reports before ending the session.

In [ ]:
import datetime
import json
import os
from pathlib import Path
import signal
import subprocess
import sys
import uuid
import zipfile

SOURCE_COMMIT = "fac8b8da180b6bca2ce7dc6267d850bba551526b"
REPOSITORY = "https://github.com/CCC917111/geospatial-mamba-deforestation.git"
BATCH_SIZES = [1, 2, 4, 8]
ROUNDS = 3
STEPS = 30
WARMUP = 5
PROFILE_STEPS = 3
WEIGHTS = None  # Optional full-default-model checkpoint; None uses seeded initialization.
SAVE_TO_DRIVE = False
SHOW_PROFILE = False  # Optional viewer installation happens only after measurements.

WORKSPACE = Path("/content/geospatial-batch-study")
WORKSPACE.mkdir(parents=True, exist_ok=True)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    report_base = Path("/content/drive/MyDrive/geospatial-mamba/batch-studies")
else:
    report_base = WORKSPACE
report_base.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
identifier = stamp + "-" + uuid.uuid4().hex[:8]
SESSION_ROOT = report_base / ("batch-study-" + identifier)
SESSION_ROOT.mkdir(exist_ok=False)
RUN_ROOT = SESSION_ROOT / "study"
SOURCE_ROOT = WORKSPACE / ("source-" + identifier)
EXPORT_BASE = report_base
os.environ.setdefault("MPLCONFIGDIR", str(WORKSPACE / "cache/matplotlib"))
os.environ.setdefault("KERAS_HOME", str(WORKSPACE / "cache/keras"))
os.environ.setdefault("XDG_CACHE_HOME", str(WORKSPACE / "cache"))
print("Reports:", SESSION_ROOT)
(SESSION_ROOT / "packages.txt").write_text(subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True))

With `SAVE_TO_DRIVE=False`, `/content` is temporary. Enable Drive before
starting if you want reports and traces written there; only successfully
written files survive runtime reclamation. A rerun creates a fresh session;
interrupted training is restarted from the shared initial weights.

In [ ]:
subprocess.run(["git", "clone", "--no-checkout", REPOSITORY, str(SOURCE_ROOT)], check=True)
subprocess.run(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=SOURCE_ROOT, check=True)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=SOURCE_ROOT, text=True).strip()
dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=SOURCE_ROOT, text=True)
if head != SOURCE_COMMIT or dirty:
    raise RuntimeError("Source revision mismatch or modified checkout")
(SESSION_ROOT / "source-revision.json").write_text(json.dumps({
    "commit": head, "batch_sizes": BATCH_SIZES, "rounds": ROUNDS,
    "steps": STEPS, "warmup": WARMUP, "profile_steps": PROFILE_STEPS,
}, indent=2))

In [ ]:
def execute_with_cleanup(argv, cwd, log):
    process = subprocess.Popen(argv, cwd=cwd, stdout=log,
                               stderr=subprocess.STDOUT, text=True,
                               start_new_session=True)
    try:
        return_code = process.wait()
    finally:
        # The process group includes the driver and its current GPU worker.
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            pass
        finally:
            # Reap descendants even if the driver has already exited.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, argv)



def bundle_downloads(directory, manifest, limit=28 * 1024 * 1024):
    names = manifest["downloads"]
    total = sum((directory / name).stat().st_size for name in names)
    if total + 1024 * len(names) >= limit:
        return names
    destination = directory / "batch-study-bundle.zip"
    with zipfile.ZipFile(destination, "x", zipfile.ZIP_STORED) as archive:
        for name in names:
            archive.write(directory / name, name)
    if destination.stat().st_size >= 32 * 1024 * 1024:
        destination.unlink()
        return names
    return [destination.name]


def export_reports():
    destination = EXPORT_BASE / (SESSION_ROOT.name + "-export-" + uuid.uuid4().hex[:6])
    subprocess.run([sys.executable, str(SOURCE_ROOT / "scripts/archive_profiles.py"),
                    "--run-root", str(SESSION_ROOT), "--output-dir", str(destination)],
                   check=True, stdout=subprocess.DEVNULL)
    manifest = json.loads((destination / "profile-archives.json").read_text())
    names = bundle_downloads(destination, manifest)
    print("Download files:", [str(destination / name) for name in names])
    return destination, names

In [ ]:
command = [sys.executable, str(SOURCE_ROOT / "scripts/study_batches.py"),
           "--source-root", str(SOURCE_ROOT), "--run-root", str(RUN_ROOT),
           "--device", "/GPU:0", "--batch-sizes", *map(str, BATCH_SIZES),
           "--rounds", str(ROUNDS), "--steps", str(STEPS), "--warmup", str(WARMUP),
           "--profile-steps", str(PROFILE_STEPS),
           "--training-precision", "mixed_float16"]
if WEIGHTS is not None:
    command += ["--weights", str(Path(WEIGHTS).expanduser().resolve())]
(SESSION_ROOT / "command.json").write_text(json.dumps(command, indent=2))
print("Running batch study. Stage logs:", SESSION_ROOT / "driver.log")
try:
    with (SESSION_ROOT / "driver.log").open("x") as log:
        execute_with_cleanup(command, cwd=SOURCE_ROOT, log=log)
except KeyboardInterrupt:
    (SESSION_ROOT / "interrupted.json").write_text(json.dumps({
        "status": "interrupted", "scope": "Driver and GPU workers stopped before export."
    }, indent=2))
    raise
except subprocess.CalledProcessError:
    print((SESSION_ROOT / "driver.log").read_text()[-6000:])
    print("Run the download cell below to save the failure report and available traces.")
    raise
finally:
    EXPORT_DIRECTORY, DOWNLOADS = export_reports()

In [ ]:
from IPython.display import Markdown, display

execution_path = RUN_ROOT / "execution.json"
if execution_path.is_file():
    print("Execution status:", json.loads(execution_path.read_text())["status"])
report_path = RUN_ROOT / "study.json"
if report_path.is_file():
    report = json.loads(report_path.read_text())
    lines = ["| Workload | Batch | Status | Batch median (ms) | Amortized ms/tile | Tiles/s | Allocator peak (MiB) |",
             "|---|---:|---|---:|---:|---:|---:|"]
    for mode, values in report["summary"].items():
        for case in values["batches"]:
            batch = case["batch_size"]
            if case["status"] != "passed":
                lines.append(f"| {mode} | {batch} | {case['status']} | — | — | — | — |")
                continue
            peak = case["median_allocator_peak_bytes"]
            peak_text = f"{peak / 2**20:.2f}" if peak is not None else "unavailable"
            lines.append(f"| {mode} | {batch} | passed | {case['median_seconds'] * 1000:.2f} | "
                         f"{case['amortized_median_seconds_per_tile'] * 1000:.2f} | "
                         f"{case['tiles_per_second']:.2f} | {peak_text} |")
    display(Markdown("\n".join(lines)))
    print("Study status:", report["status"])
    print("GPU trace coverage:", report.get("gpu_trace_coverage", "not verified: study incomplete"))
else:
    print("Timing did not complete. Download the available failure reports below.")


Download the bundle below and return it for review. It contains the raw timing
reports, process status, package versions, fixture identities and complete
profile traces. Input arrays and checkpoint files remain in the runtime.

When the complete bundle would exceed the transfer budget, the cell downloads
`metadata.zip`, `profile-archives.json` and numbered trace ZIPs instead. Each
ZIP is below 32 MiB. Keep and return every part; metadata alone cannot show
GPU kernel bottlenecks. Traces are split losslessly and verified when restored.
If the study failed, this cell can still download the available evidence.

In [ ]:
from google.colab import files

if "EXPORT_DIRECTORY" not in globals():
    EXPORT_DIRECTORY, DOWNLOADS = export_reports()
for name in DOWNLOADS:
    files.download(str(EXPORT_DIRECTORY / name))

## Optional: inspect the GPU capture

Set `SHOW_PROFILE=True` to run the next cell **after measurements**. It installs
[XProf](https://openxla.org/xprof) with constraints that preserve the measured
TensorFlow, TensorBoard, Keras, NumPy, protobuf and ml-dtypes versions. A viewer
dependency conflict stops this optional cell without changing the results.
Exact viewer compatibility depends on the Colab environment.

Open the **Profile** tab. A real GPU capture must show **GPU stream tracks**
in Trace Viewer and populated **GPU Kernel Stats**. Export Kernel Stats as CSV
and include it with your reports if possible. Check host launch gaps, recurrent
SSM scopes and convolution kernels before attributing the bottleneck. Existing
layer names are recorded in each `profile-*.json` under `layer_scopes`.

A saved `.xplane.pb` can contain only host events. If GPU tracks are absent,
retain the logs and inspect CUPTI loading or privilege errors; do not treat
host-only events as GPU timings. The notebook does not change CUDA or system
privileges. See [capture troubleshooting](https://openxla.org/xprof/capturing_profiles)
and [GPU Kernel Stats](https://openxla.org/xprof/gpu_kernel_stats).

In [ ]:
if SHOW_PROFILE:
    import importlib.metadata as metadata
    import shlex

    protected = {}
    for package in ("tensorflow", "tensorboard", "keras", "numpy", "protobuf", "ml-dtypes"):
        try:
            protected[package] = metadata.version(package)
        except metadata.PackageNotFoundError:
            pass
    constraints = SESSION_ROOT / "viewer-constraints.txt"
    constraints.write_text("".join(f"{name}=={version}\n" for name, version in protected.items()))
    subprocess.run([sys.executable, "-m", "pip", "install", "--constraint",
                    str(constraints), "xprof"], check=True)
    (SESSION_ROOT / "packages-viewer.txt").write_text(subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"], text=True))
    EXPORT_DIRECTORY, DOWNLOADS = export_reports()
    print("Rerun the download cell to include the recorded viewer environment.")
    from tensorboard import notebook
    notebook.start("--logdir " + shlex.quote(str(RUN_ROOT / "profiles")))
else:
    print("Optional viewer skipped. Reports and raw traces are already exported.")

## Restore downloaded traces elsewhere

Unzip `batch-study-bundle.zip` if a single bundle was downloaded. Keep
`profile-archives.json` and all numbered trace ZIPs in the same directory:

```bash
python scripts/archive_profiles.py --restore downloads/profile-archives.json \
  --archive-dir downloads --destination restored-study
```

The new destination contains the original `study/profiles/...` directory
structure. Point the viewer at `restored-study/study/profiles`. Restore checks
every archive, chunk and complete trace hash, and refuses to overwrite files.
The [profiling guide](https://github.com/CCC917111/geospatial-mamba-deforestation/blob/main/docs/gpu-profiling.md)
explains the protocol and how to read the results.